## RUN - 2 - Movement vectors

### Notes

- **Information on input**
    - `Ectoderm-data_clean.pkl`; dictionary of cleaned Ectoderm data generated during preprocessing
    - `Myeloid-data_clean.pkl`; dictionary of cleaned Myeloid data generated during preprocessing
    - Data structure: 
        - `'norm' vs 'dilu'` for normal vs diluted (1:1200) fibronectin conditions
            - `'Pos{ddd}'` for each explants sample
                - Pandas df of shape `tracks X (f, t, y, x)`
                - with units: `[f] = [1]`, `[t] = [min]`, `[y, x] = [microns]`
    - Tracking was done by Anh with StarDist for the ectoderm and manually in ImageJ for myeloid cells


* **Pre-requisites**
    - The data must have been preprocessed with `RUN - 1 - Preprocessing.ipynb`


- **Content of this notebook**
    1. Load the standardized data
    2. Extract additional spatial information (cluster centers, radii, etc.)
    3. Extract movement vectors from consecutive positions
    4. Interpolate movement vectors from cell neighborhood
    5. Save extracted data for next steps
    

* **Outputs of this notebook**
    - `ecto_data[conditions][positions]`; pandas dfs of shape `tracks X (f, t, y, x, ...)*`
    - `myel_data[conditions][positions]`; pandas dfs of shape `tracks X (f, t, y, x, ...)*`
    - `*` Details on df columns:
        - `f, t, y, x`: frames `[1]`, times `[min]`, positions `[microns]`
        - `cen_y, cen_x, y_rel, x_rel, r, clust_r`: cluster center, relative positions, radial distances, cluster size `[microns]`
        - `vy, vx, vy_itp, vx_itp`: velocities, locally interpolated ectoderm velocities `[microns/min]`

### Prep

In [ ]:
### Imports

%load_ext autoreload
%autoreload 2

import os, warnings, pickle

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from ipywidgets import interact

import scipy.spatial.distance as dist

import sys; sys.path.insert(0, '..')
import tracking_analysis.pcl_tools as pcl
from tracking_analysis.utilities import savebutton

In [ ]:
### Parameters

# Overwrite or dry run
save_outputs = True

# Units
pxl_res  = 1.5152  # [microns]
time_res = 5       # [min]

# Vector interpolation
#sigma = 50  # Sigma for Gaussian; initial guess
sigma = 40   # Sigma for Gaussian; based on testing

In [ ]:
### Data locations

top_path = r"..\Data\ex_vivo"

ecto_file = r"Ectoderm-data_clean.pkl"
myel_file = r"Myeloid-data_clean.pkl"

In [ ]:
### Load the data

# Ectoderm
with open(os.path.join(top_path, ecto_file), "rb") as infile:
    ecto_data = pickle.load(infile)

# Myeloid
with open(os.path.join(top_path, myel_file), "rb") as infile:
    myel_data = pickle.load(infile)

# Report
print("\nEctoderm data:")
display(ecto_data['norm'].keys())
display(ecto_data['dilu'].keys())
display(ecto_data['norm']['Pos001'].head())
display(ecto_data['norm']['Pos001'].shape)

print("\nMyeloid data:")
display(myel_data['norm'].keys())
display(myel_data['dilu'].keys())
display(myel_data['norm']['Pos001'].head())
display(myel_data['norm']['Pos001'].shape)

### Extract additional spatial information

This includes: 

- Cluster (ectoderm) centroid; `cen_y, cen_x`
- Cell relative coordinates; `y_rel, x_rel`
- Cell radial coordinates;  `r`
- Cluster (ectoderm) radius; `clust_r`

In [ ]:
### Extract additional spatial information

# For each condition c and position p...
for c in ecto_data.keys():
    for p in ecto_data[c].keys():
        
        # Get cluster center (ectoderm centroids)
        ecto_data[c][p][["cen_y", "cen_x"]] = ecto_data[c][p].groupby("f")[["y", "x"]].transform("mean")
        myel_data[c][p] = myel_data[c][p].join(ecto_data[c][p].groupby("f")[["cen_y", "cen_x"]].first(), on="f")
        
        # Get cell coordinates relative to cluster center
        ecto_data[c][p][["y_rel", "x_rel"]] = ecto_data[c][p][["y", "x"]].values - ecto_data[c][p][["cen_y", "cen_x"]].values
        myel_data[c][p][["y_rel", "x_rel"]] = myel_data[c][p][["y", "x"]].values - myel_data[c][p][["cen_y", "cen_x"]].values
        
        # Get cell radial coordinates (distance from center)
        ecto_data[c][p]["r"] = np.sqrt(np.sum(ecto_data[c][p][["y_rel", "x_rel"]].values**2, axis=1))
        myel_data[c][p]["r"] = np.sqrt(np.sum(myel_data[c][p][["y_rel", "x_rel"]].values**2, axis=1))
        
        # Approximate cluster radii (97th percentile of ectoderm cell radii)
        ecto_data[c][p]["clust_r"] = ecto_data[c][p].groupby("f")["r"].transform("quantile", 0.97)
        myel_data[c][p] = myel_data[c][p].join(ecto_data[c][p].groupby("f")["clust_r"].first(), on="f")

### Extract movement vectors

In [ ]:
### Extract movement vectors

# Function to get difference vectors (w/ last vector forced to zero)
def get_diffvecs(df):
    df[['vy', 'vx']] = np.diff(
        df[['y', 'x']], axis=0, 
        append=df[['y', 'x']].iloc[-1].values.reshape(1, -1)) / time_res
    return df

# For each condition c and position p...
for c in ecto_data.keys():
    for p in ecto_data[c].keys():
        
        # Get vectors
        ecto_df = ecto_data[c][p].groupby('track').apply(get_diffvecs).droplevel(0)
        myel_df = myel_data[c][p].groupby('track').apply(get_diffvecs).droplevel(0)

        # Remove last time point of each track bc vectors are unavailable
        ecto_last_tp_mask = ecto_df.groupby('track')['f'].transform("max") == ecto_df['f']
        ecto_data[c][p]   = ecto_df.loc[~ecto_last_tp_mask, :]
        myel_last_tp_mask = myel_df.groupby('track')['f'].transform("max") == myel_df['f']
        myel_data[c][p]   = myel_df.loc[~myel_last_tp_mask, :]

In [ ]:
### Visualize resulting vectors

@interact(condition=ecto_data.keys())
def show_by_condition(condition='norm'):
    @interact(position=ecto_data[condition].keys())
    @savebutton
    def show_by_position(position=list(ecto_data[condition].keys())[0]):

        fig, ax = plt.subplots(1, 3, figsize=(10, 2.5))
        
        ax[0].hist(ecto_data[condition][position]['vy'], bins=200)
        ax[1].hist(myel_data[condition][position]['vy'], bins=200)
        
        ex_tp0 = myel_data[condition][position].loc[myel_data[condition][position]['f']==100]
        ex_tp1 = myel_data[condition][position].loc[myel_data[condition][position]['f']==101]
        ax[2].scatter(ex_tp0['x'], ex_tp0['y'], s=5, alpha=0.5, label='tp=100')
        ax[2].scatter(ex_tp1['x'], ex_tp1['y'], s=5, alpha=0.5, label='tp=101')
        ax[2].quiver(ex_tp0['x'], ex_tp0['y'], ex_tp0['vx']*time_res, ex_tp0['vy']*time_res,
                     angles='xy', scale_units='xy', scale=1.0, 
                     color='k', alpha=1.0, width=0.003)
        
        xmin, xmax = ax[2].get_xlim()
        ymin, ymax = ax[2].get_ylim()
        ax[2].set_xlim(xmin, xmin + (xmax - xmin) / 2)
        ax[2].set_ylim(ymin, ymin + (ymax - ymin) / 2)
        
        ax[0].set_title(f'ectoderm ({condition} - {position})')
        ax[1].set_title(f'myeloid ({condition} - {position})')
        ax[2].set_title(f'example vecs ({condition} - {position})')
        ax[0].set_xlabel(r'vy $\mathrm{[\mu m/\min]}$')
        ax[1].set_xlabel(r'vy $\mathrm{[\mu m/\min]}$')
        ax[2].set_xlabel('x')
        ax[0].set_ylabel('count')
        ax[2].set_ylabel('y')
        
        plt.tight_layout()

### Movement vector interpolation from neighborhood

In [ ]:
### Interpolate movement vectors from neighborhood

# For each condition c and position p...
for c in ecto_data.keys():
    for p in ecto_data[c].keys():
    
        # Point to relevant data
        ecto_df = ecto_data[c][p]
        myel_df = myel_data[c][p]

        # Prep output container
        ecto_df[['vy_itp', 'vx_itp']] = np.nan
        myel_df[['vy_itp', 'vx_itp']] = np.nan

        # For each frame...
        for f in range(ecto_df['f'].max()+1):

            # Generate frame mask
            ecto_fmask = (ecto_df['f'] == f).values
            myel_fmask = (myel_df['f'] == f).values

            # Compute mutual distances
            ecto_dists = dist.cdist(
                ecto_df.loc[ecto_fmask, ['y', 'x']].values, 
                ecto_df.loc[ecto_fmask, ['y', 'x']].values)
            myel_dists = dist.cdist(
                ecto_df.loc[ecto_fmask, ['y', 'x']].values, 
                myel_df.loc[myel_fmask, ['y', 'x']].values)

            # Run interpolation
            ecto_df.loc[ecto_fmask, ['vy_itp', 'vx_itp']] = pcl.pcl_gaussian_interp(
                ecto_dists, ecto_df.loc[ecto_fmask, ['vy', 'vx']].values, sigma=sigma)
            if myel_dists.shape[1] > 0:
                myel_df.loc[myel_fmask, ['vy_itp', 'vx_itp']] = pcl.pcl_gaussian_interp(
                    myel_dists, ecto_df.loc[ecto_fmask, ['vy', 'vx']].values, sigma=sigma)

In [ ]:
### Check that there are no missing values

for c in ecto_data.keys():
    for p in ecto_data[c].keys():
        ecto_good = ~ecto_data[c][p][['vy_itp', 'vx_itp']].isnull().values.any()
        myel_good = ~myel_data[c][p][['vy_itp', 'vx_itp']].isnull().values.any()
        print(f"{c}, {p} -- ecto: {ecto_good}, myel: {myel_good}")

In [ ]:
### Visualize interpolated vector values

@interact(condition=ecto_data.keys())
def show_by_condition(condition='norm'):
    @interact(position=ecto_data[condition].keys())
    @savebutton
    def show_by_position(position=list(ecto_data[condition].keys())[0]):

        fig, ax = plt.subplots(1, 2, figsize=(6.6, 2.5))
        
        _, bins, _ = ax[0].hist(
            ecto_data[condition][position]['vy'], 
            bins=200, alpha=0.5, label='data')
        ax[0].hist(
            ecto_data[condition][position]['vy_itp'], 
            bins=bins, alpha=0.5, label='itp')
        
        _, bins, _ = ax[1].hist(
            myel_data[condition][position]['vy'], 
            bins=200, alpha=0.5, label='data')
        ax[1].hist(
            myel_data[condition][position]['vy_itp'], 
            bins=bins, alpha=0.5, label='itp')
        
        ax[0].legend()
        ax[0].set_title(f'ectoderm ({condition} - {position})')
        ax[1].set_title(f'myeloid ({condition} - {position})')
        ax[0].set_xlabel('vy')
        ax[1].set_xlabel('vy')
        ax[0].set_ylabel('count')
        
        plt.tight_layout()

In [ ]:
### Visualize interpolated vectors

@interact(condition=ecto_data.keys())
def show_by_condition(condition='norm'):
    @interact(position=ecto_data[condition].keys())
    def show_by_position(position=list(ecto_data[condition].keys())[0]):
        @interact(frame=(0, ecto_data[condition][position]['f'].max(), 1), 
                  scale=(0.01, 0.1, 0.01))
        @savebutton
        def show_vectors(frame=ecto_data[condition][position]['f'].max()//2, 
                         scale=0.15):
            
            # Select relevant data
            ecto_df = ecto_data[condition][position]
            myel_df = myel_data[condition][position]
            
            # Create frame masks
            ecto_fmask = (ecto_df['f'] == frame).values
            myel_fmask = (myel_df['f'] == frame).values
            myel_fmask_p1 = (myel_df['f'] == frame+1).values

            # Prep figure
            fig, ax = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)

            # Plot observed vectors
            ax[0].quiver(ecto_df.loc[ecto_fmask, 'x'],  ecto_df.loc[ecto_fmask, 'y'],
                         ecto_df.loc[ecto_fmask, 'vx'], ecto_df.loc[ecto_fmask, 'vy'],
                         angles='xy', scale_units='xy', scale=scale, 
                         color='black', alpha=0.6, width=0.003)
            ax[1].quiver(myel_df.loc[myel_fmask, 'x'],  myel_df.loc[myel_fmask, 'y'],
                         myel_df.loc[myel_fmask, 'vx'], myel_df.loc[myel_fmask, 'vy'],
                         angles='xy', scale_units='xy', scale=scale, 
                         color='black', alpha=0.6, width=0.003)

            # Plot interpolated vectors
            ax[0].quiver(ecto_df.loc[ecto_fmask, 'x'], ecto_df.loc[ecto_fmask, 'y'],
                         ecto_df.loc[ecto_fmask, 'vx_itp'], ecto_df.loc[ecto_fmask, 'vy_itp'],
                         angles='xy', scale_units='xy', scale=scale,
                         color='red', alpha=0.6, width=0.003)
            ax[1].quiver(myel_df.loc[myel_fmask, 'x'], myel_df.loc[myel_fmask, 'y'],
                         myel_df.loc[myel_fmask, 'vx_itp'], myel_df.loc[myel_fmask, 'vy_itp'],
                         angles='xy', scale_units='xy', scale=scale, 
                         color='red', alpha=0.6, width=0.003)

            # Plot source points
            ax[0].scatter(ecto_df.loc[ecto_fmask, 'x'], ecto_df.loc[ecto_fmask, 'y'], 
                          c='darkblue', s=5, alpha=1.0)
            ax[1].scatter(myel_df.loc[myel_fmask, 'x'], myel_df.loc[myel_fmask, 'y'], 
                          c='darkblue', s=5, alpha=1.0)

            # Axis limits
            ax[0].set_xlim([0, 800])
            ax[0].set_ylim([700, 1500])

            # Labels
            ax[0].set_title(f'Ectoderm (t={frame*time_res}min)')
            ax[1].set_title(f'Myeloid (t={frame*time_res}min)')

            # Finish
            plt.tight_layout()

### Save the results

In [ ]:
if save_outputs:

    # Ectoderm
    with open(os.path.join(top_path, "Ectoderm-data_clean_vec.pkl"), "wb") as outfile:
        pickle.dump(ecto_data, outfile, protocol=pickle.HIGHEST_PROTOCOL)

    # Myeloid
    with open(os.path.join(top_path, "Myeloid-data_clean_vec.pkl"), "wb") as outfile:
        pickle.dump(myel_data, outfile, protocol=pickle.HIGHEST_PROTOCOL)